# LIME 

Este notebook:
- Carga un **modelo preentrenado de torchvision** (ResNet50).
- Aplica **LIME** para explicar la **clase predicha** en **todas las imágenes** de una carpeta.
- Guarda resultados en `output_lime/` como PNG.


In [1]:
# (Opcional) Si te falta algo en tu entorno, descomenta e instala:
# !pip install -U torch torchvision pillow matplotlib numpy
# !pip install -U lime scikit-image

import os
from pathlib import Path
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

if torch.cuda.is_available():
    device = torch.device("cuda")      # o "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple Silicon
else:
    device = torch.device("cpu")


print("Device:", device)


Device: mps


In [3]:
# Imports LIME + utilidades de segmentación/visualización

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
except Exception as e:
    raise ImportError(
        "No puedo importar LIME o scikit-image. Instala con: pip install lime scikit-image"
    ) from e


In [4]:
# Cargar modelo torchvision (ResNet50 preentrenado)

try:
    from torchvision.models import resnet50, ResNet50_Weights
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    imagenet_labels = weights.meta.get("categories", None)
except Exception:
    from torchvision.models import resnet50
    model = resnet50(pretrained=True)
    imagenet_labels = None

model = model.to(device).eval()
print("Modelo listo:", model.__class__.__name__)


Modelo listo: ResNet


In [ ]:
# Utilidades: carga segura de imágenes y preprocesado (ImageNet)

IM_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def safe_load_rgb(path: str, size=IM_SIZE):
    """Carga una imagen (segura) en RGB, la redimensiona y devuelve array float [0,1] HxWx3."""
    try:
        with Image.open(path) as im:
            im.verify()
        pil = Image.open(path).convert("RGB")
        pil = pil.resize((size, size))
        arr = np.asarray(pil).astype(np.float32) / 255.0
        return pil, arr
    except (UnidentifiedImageError, OSError) as e:
        print(f"Saltando archivo no válido: {path} | {e}")
        return None, None

def preprocess_batch(images_0_1: np.ndarray) -> torch.Tensor:
    """images_0_1: NxHxWx3 float [0,1] -> tensor Nx3xHxW normalizado."""
    x = (images_0_1 - MEAN) / STD
    x = torch.from_numpy(x).permute(0, 3, 1, 2).float()
    return x.to(device)


In [9]:
# Función de predicción para LIME (devuelve probabilidades NxC)

@torch.no_grad()
def classifier_fn(images: np.ndarray) -> np.ndarray:
    """images: NxHxWx3 (valores en [0,1]) -> probs Nx1000"""
    if images.dtype != np.float32:
        images = images.astype(np.float32)

    x = preprocess_batch(images)  # Nx3xHxW
    logits = model(x)
    probs = F.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

def top1_label(arr_0_1: np.ndarray):
    probs = classifier_fn(arr_0_1[None, ...])[0]
    idx = int(np.argmax(probs))
    score = float(probs[idx])
    name = imagenet_labels[idx] if imagenet_labels is not None else str(idx)
    return idx, name, score


In [10]:
# CONFIG: carpeta de entrada y salida
INPUT_DIR = Path("../imagenes/original")    
OUTPUT_DIR = Path("../imagenes/output_lime")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Opcional: limitar cuántas procesa (None = todas)
MAX_IMAGES = None

# Parámetros LIME
LIME_NUM_SAMPLES = 2000   # más alto = más estable, más lento
NUM_FEATURES = 8         # superpíxeles a mostrar
POSITIVE_ONLY = True     # mostrar solo contribuciones positivas
HIDE_REST = False        # si True, oculta todo excepto los superpíxeles seleccionados

print("INPUT_DIR:", INPUT_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


INPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/original
OUTPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/output_lime


In [11]:
# Ejecutar LIME sobre todas las imágenes de la carpeta

paths = [p for p in sorted(INPUT_DIR.rglob("*")) if p.suffix.lower() in EXTS]
if MAX_IMAGES is not None:
    paths = paths[:MAX_IMAGES]

print("Imágenes encontradas:", len(paths))
if len(paths) == 0:
    raise FileNotFoundError(f"No encontré imágenes en {INPUT_DIR}. Revisa INPUT_DIR.")

explainer = lime_image.LimeImageExplainer()
processed = 0

for i, p in enumerate(paths, 1):
    pil, img_0_1 = safe_load_rgb(str(p), size=IM_SIZE)
    if pil is None:
        continue

    # Clase objetivo = top-1 del modelo
    cls_idx, cls_name, cls_prob = top1_label(img_0_1)

    # LIME necesita imagen en formato HxWx3 (puede ser float [0,1])
    explanation = explainer.explain_instance(
        img_0_1,
        classifier_fn,
        top_labels=1,
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES
    )

    # Obtener máscara y visualización
    temp, mask = explanation.get_image_and_mask(
        label=cls_idx,
        positive_only=POSITIVE_ONLY,
        num_features=NUM_FEATURES,
        hide_rest=HIDE_REST
    )

    # temp está en float [0,1] si la entrada lo estaba
    # Dibujamos límites de superpíxeles seleccionados
    lime_vis = mark_boundaries(temp, mask)

    out_name = OUTPUT_DIR / f"{p.stem}_lime.png"

    fig = plt.figure(figsize=(10, 3))
    ax1 = plt.subplot(1, 3, 1)
    ax1.imshow(img_0_1)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(1, 3, 2)
    ax2.imshow(mask, cmap="gray")
    ax2.set_title("Máscara (superpíxeles)")
    ax2.axis("off")

    ax3 = plt.subplot(1, 3, 3)
    ax3.imshow(lime_vis)
    ax3.set_title(f"LIME\n{cls_name} ({cls_prob:.2f})")
    ax3.axis("off")

    plt.tight_layout()
    fig.savefig(out_name, bbox_inches="tight")
    plt.close(fig)

    processed += 1
    if processed % 5 == 0 or i == len(paths):
        print(f"Procesadas: {processed} | Último guardado: {out_name}")

print("Revisa la carpeta output_lime/")


Imágenes encontradas: 11


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

...

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Procesadas: 10 | Último guardado: ../imagenes/output_lime/image10_lime.png


  0%|          | 0/2000 [00:00<?, ?it/s]

Procesadas: 11 | Último guardado: ../imagenes/output_lime/image11_lime.png
Revisa la carpeta output_lime/
